In [56]:
import pandas as pd
import numpy as np

In [57]:
data = pd.read_csv('../course_lead_scoring_2026.csv')

In [58]:
data.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


### **Подготовка данных.**

In [59]:
#кол-во пропущенных значений
data.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [60]:
#тип данных каждого столбца
data.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [61]:
cat_cols = data.select_dtypes(include=["object", "str"]).columns
num_cols = data.select_dtypes(include="number").columns

#заполним пропущенные значения
data[cat_cols] = data[cat_cols].fillna("NA")
data[num_cols] = data[num_cols].fillna(0.0)

In [62]:
data.isnull().sum().sum()

np.int64(0)

### **Задание 1.**
Найдем моду столбца industry.

In [63]:
data["industry"].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

### **Задание 2.**
Корреляционная матрица.

In [64]:
#матрица корреляций
corr_matrix = data.corr(numeric_only=True)
corr_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [65]:
#пары, которые нас интересуют
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count")
]

results = [] #список словарей

#соберем нужные нам пары
for col1, col2 in pairs:
    corr = corr_matrix.loc[col1, col2]
    results.append({
        "Пара признаков": f"{col1} и {col2}",
        "Корреляция": corr
    })

#превратим это в таблицу
result_df = pd.DataFrame(results)
#отсортируем
result_df = result_df.sort_values(
    by="Корреляция", ascending=False
)

result_df

,Пара признаков,Корреляция
0,interaction_count и lead_score,0.915746
1,number_of_courses_viewed и lead_score,0.757204
2,number_of_courses_viewed и interaction_count,0.721609
3,annual_income и interaction_count,0.122842


### **Разделим данные.**

In [66]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(data, test_size=0.2, random_state=42)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

### **Задание 3.**
Mutual Information (MI) между признаками и таргетом.

**Mutual Information (MI)** — _показывает, насколько один признак помогает нам лучше понять другой. Она удобна для категориальных признаков, потому что умеет находить связь между разными категориями без необходимости превращать их в числа._

In [67]:
from sklearn.metrics import mutual_info_score

In [73]:
mi = {}

for col in cat_cols:
    mi[col] = round(mutual_info_score(df_train[col], df_train['converted']), 2)

mi_df = pd.DataFrame(mi.items(), columns=['feature', 'MI'])
mi_df.sort_values(by='MI', ascending=False)
mi_df

,feature,MI
0,lead_source,0.03
1,industry,0.00
2,employment_status,0.02
3,location,0.00


### **Задание 4.**
Обучим модель, которая будет предсказывать, зарегистрируется ли клиент на платформе (converted).

In [74]:
from sklearn.feature_extraction import DictVectorizer
#DictVectorizer автоматически применяет One-Hot Encoding к категориальным признакам и сохраняет числовые признаки в числовом виде.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

_закодируем категориальные признаки в обучающих данных_

In [77]:
#отделим таргет
y_train = df_train['converted']
X_train = df_train.drop(columns=['converted'])

In [83]:
#создадим словарь из каждой строки
train_dict = X_train.to_dict(orient='records')

#преобразователь для кодирования
dv = DictVectorizer(sparse=False)
#изучим признаки и сразу преобразуем
X_train_encoded = dv.fit_transform(train_dict)

_закодируем категориальные признаки в валидационных данных_

In [81]:
#отделим таргет
y_val = df_val['converted']
X_val = df_val.drop(columns=['converted'])

In [84]:
#словарь из каждой строки
val_dict = X_val.to_dict(orient='records')

#преобразуем
X_val_encoded = dv.transform(val_dict)

_Обучим логистическую регрессию._

In [97]:
model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)
model.fit(X_train_encoded, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [98]:
#предсказание модели
y_pred = model.predict(X_val_encoded)

In [99]:
#доля правильных предсказаний модели
accuracy_score(y_val, y_pred)

0.645

### **Задание 5.**
Выясним, какой признак сильнее всего влияет на качество нашей логистической регрессии.

In [100]:
#сохраним исходную точность
original_accuracy = accuracy_score(y_val, y_pred)

0.645

In [101]:
#признаки для исключения
features_to_remove = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

In [102]:
#будем смотреть на разные результаты
results = {}

for feature in features_to_remove:
    #удалим
    X_train_reduced = X_train.drop(columns=[feature])
    X_val_reduced = X_val.drop(columns=[feature])

    #для кодирования признаков
    dv_reduced = DictVectorizer(sparse=False)

    #кодируем признаки
    X_train_reduced_encoded = dv_reduced.fit_transform(
        X_train_reduced.to_dict(orient='records')
    )
    X_val_reduced_encoded = dv_reduced.transform(
        X_val_reduced.to_dict(orient='records')
    )

    #модель
    model = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    #обучаем модель
    model.fit(X_train_reduced_encoded, y_train)
    y_pred_reduced = model.predict(X_val_reduced_encoded)

    #считаем результаты
    accuracy = accuracy_score(y_val, y_pred_reduced)
    difference = original_accuracy - accuracy

    results[feature] = difference

results

{'lead_source': 0.0030000000000000027,
 'number_of_courses_viewed': 0.0020000000000000018,
 'interaction_count': 0.04400000000000004}

### **Задание 6.**
Исследуем, как параметр C влияет на качество логистической регрессии.

In [103]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

In [104]:
results = {}

for C in C_values:
    #модель
    model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )

    #обучим
    model.fit(X_train_encoded, y_train)

    #предсказание
    y_pred = model.predict(X_val_encoded)

    #результат
    accuracy = accuracy_score(y_val, y_pred)
    results[C] = round(accuracy, 3)

for C, accuracy in results.items():
    print(f'C = {C}: accuracy = {accuracy:.3f}')

C = 1e-06: accuracy = 0.598
C = 1e-05: accuracy = 0.598
C = 0.0001: accuracy = 0.613
C = 0.001: accuracy = 0.645
